In [1]:
# R1b_correspondence_german_heloc.ipynb  (revision stage 1, generalization)
# Same reconstructed test as R1 (multivariate dummy-logit projection slope vs
# WoE-logit beta_hat; componentwise boosting -> dummy MLE) on German (nominal-heavy)
# and HELOC (informative missing codes). Binning + IV selection INSIDE each fold.
# German: categoricals -> per-level bins; numerics -> monotone bins.
# HELOC: special codes -7/-8/-9 as own bins; numerics monotone; drop all-(-9) rows.
# Single-cell; greyscale seaborn; dpi600 png+pdf; no captions.

import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import statsmodels.api as sm
warnings.filterwarnings('ignore')

SEED=42; IV_THRESH=0.02
sns.set_theme(style='whitegrid', context='paper'); sns.set_palette('Greys')
plt.rcParams.update({'axes.edgecolor':'0.2','grid.color':'0.85','font.size':10})
ROOT=os.path.abspath(os.path.join(os.getcwd(),'..')); DATA=os.path.join(ROOT,'data')
TABLES=os.path.join(ROOT,'results','tables'); FIGS=os.path.join(ROOT,'results','figures')
def savefig(fig,name):
    for e in ('png','pdf'): fig.savefig(os.path.join(FIGS,name+'.'+e),dpi=600,bbox_inches='tight')
    plt.close(fig)
SPECIAL=[-7,-8,-9]

# ===================================================================
# generic fold engine given a per-fold binning function -> maps[c]=list of recs
#   rec: {'woe','n','good','bad', + kind-specific} ; needs apply(idx) mapping
# ===================================================================
def woe_of(g,b,TG,TB): gr=(g+0.5)/(TG+0.5); br=(b+0.5)/(TB+0.5); return float(np.log(br/gr))

def run_dataset(name, df, TARGET, cat_cols, num_cols, N_SPLIT, n_est_boost=300):
    y=df[TARGET].values.astype(int); FEAT=cat_cols+num_cols
    def tree_edges(x,yv,max_leaf=5,min_frac=0.05):
        m=~x.isin(SPECIAL) & x.notna(); xx=x[m].values.reshape(-1,1).astype(float); yy=yv[m.values]
        if len(np.unique(xx))<=2: return []
        t=DecisionTreeClassifier(max_leaf_nodes=max_leaf,min_samples_leaf=int(min_frac*len(xx)),random_state=SEED)
        t.fit(xx,yy); thr=t.tree_.threshold[t.tree_.feature>=0]; return sorted(float(v) for v in thr)
    def build_maps(Xtr,ytr):
        TG=int((ytr==0).sum()); TB=int((ytr==1).sum()); min_n=max(int(0.02*len(ytr)),10); maps={}; iv={}
        for c in cat_cols:
            recs=[]
            for lv in sorted(Xtr[c].dropna().unique()):
                sel=Xtr[c]==lv; g=int(((ytr==0)&sel.values).sum()); b=int(((ytr==1)&sel.values).sum())
                recs.append({'kind':'lvl','lv':lv,'good':g,'bad':b})
            # merge sparse levels into 'other'
            keep=[]; og=ob=0
            for r in recs:
                if r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe_of(r['good'],r['bad'],TG,TB); r['n']=r['good']+r['bad']
            maps[c]=keep
            iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        for c in num_cols:
            edges=tree_edges(Xtr[c],ytr)
            # monotone enforce
            def tbl(e):
                cuts=[-np.inf]+list(e)+[np.inf]; m=~Xtr[c].isin(SPECIAL)&Xtr[c].notna()
                idx=pd.cut(Xtr[c][m],cuts,labels=False,include_lowest=True); rr=[]
                for b in range(len(cuts)-1):
                    sel=idx==b; g=int((ytr[m.values][sel]==0).sum()); bd=int((ytr[m.values][sel]==1).sum())
                    if g+bd>0: rr.append({'kind':'num','lo':cuts[b],'hi':cuts[b+1],'good':g,'bad':bd})
                return rr
            e=list(edges)
            while e:
                rr=tbl(e); br=np.array([r['bad']/(r['good']+r['bad']) for r in rr])
                if len(br)<=2 or np.all(np.diff(br)>=-1e-12) or np.all(np.diff(br)<=1e-12): break
                d=np.diff(br); s=np.sign(np.nansum(d)); e.pop(int(np.argmax(-s*d)))
            recs=tbl(e)
            for code in SPECIAL:
                sel=Xtr[c]==code; g=int((ytr[sel.values]==0).sum()); b=int((ytr[sel.values]==1).sum())
                if g+b>0: recs.append({'kind':'code','code':code,'good':g,'bad':b})
            # NaN bin
            seln=Xtr[c].isna(); gn=int((ytr[seln.values]==0).sum()); bn=int((ytr[seln.values]==1).sum())
            if gn+bn>0: recs.append({'kind':'nan','good':gn,'bad':bn})
            # merge sparse code/nan
            keep=[]; og=ob=0
            for r in recs:
                if r['kind'] in ('code','nan') and r['good']+r['bad']<min_n: og+=r['good']; ob+=r['bad']
                else: keep.append(r)
            if og+ob>0: keep.append({'kind':'other','good':og,'bad':ob})
            for r in keep: r['woe']=woe_of(r['good'],r['bad'],TG,TB); r['n']=r['good']+r['bad']
            maps[c]=keep
            iv[c]=float(np.sum([((r['bad']+.5)/(TB+.5)-(r['good']+.5)/(TG+.5))*r['woe'] for r in keep]))
        keep=[c for c in FEAT if iv[c]>=IV_THRESH]
        return maps,keep
    def to_idx(x, recs):
        idx=np.full(len(x),-1,int)
        fin=[r for r in recs if r['kind']=='num']
        for p,r in enumerate(recs):
            if r['kind']=='num':
                sel=(~x.isin(SPECIAL)).values&x.notna().values&(x>r['lo']).values&(x<=r['hi']).values; idx[sel]=p
            elif r['kind']=='lvl': idx[(x==r['lv']).values]=p
            elif r['kind']=='code': idx[(x==r['code']).values]=p
            elif r['kind']=='nan': idx[x.isna().values]=p
        oth=[p for p,r in enumerate(recs) if r['kind']=='other']
        if oth: idx[idx==-1]=oth[0]
        else: idx[idx==-1]=0
        return idx
    def woe_design(Xdf,maps,keep):
        return np.column_stack([np.array([r['woe'] for r in maps[c]])[to_idx(Xdf[c],maps[c])] for c in keep])
    def dummy_design(Xdf,maps,keep):
        cols=[]; k=[]
        for c in keep:
            idx=to_idx(Xdf[c],maps[c]); K=len(maps[c])
            for b in range(K-1): cols.append((idx==b).astype(float)); k.append((c,b))
        return np.column_stack(cols),k

    skf=StratifiedKFold(n_splits=N_SPLIT,shuffle=True,random_state=SEED)
    rows=[]; lr=[]
    for fold,(tr,te) in enumerate(skf.split(df[FEAT],y)):
        Xtr=df.iloc[tr]; ytr=y[tr]; maps,keep=build_maps(Xtr,ytr)
        if len(keep)<2: continue
        pop={c:np.array([r['n'] for r in maps[c]],float) for c in keep}
        woeb={c:np.array([r['woe'] for r in maps[c]]) for c in keep}
        Xw=woe_design(Xtr,maps,keep)
        try: mw=sm.Logit(ytr,sm.add_constant(Xw)).fit(disp=0)
        except Exception: continue
        beta={c:mw.params[i+1] for i,c in enumerate(keep)}; ll_woe=mw.llf; k_woe=len(keep)+1
        Xd,key=dummy_design(Xtr,maps,keep)
        clf=LogisticRegression(C=5000,max_iter=5000).fit(Xd,ytr)
        p=np.clip(clf.predict_proba(Xd)[:,1],1e-12,1-1e-12)
        ll_d=float(np.sum(ytr*np.log(p)+(1-ytr)*np.log(1-p))); k_d=Xd.shape[1]+1
        gamma={c:np.zeros(len(maps[c])) for c in keep}
        for (c,b),v in zip(key,clf.coef_.ravel()): gamma[c][b]=v
        for c in keep: gamma[c]-=np.sum(pop[c]*gamma[c])/pop[c].sum()
        for c in keep:
            w=pop[c]; wv=woeb[c]-np.sum(pop[c]*woeb[c])/pop[c].sum(); den=np.sum(w*wv**2)
            slope=np.sum(w*gamma[c]*wv)/den if den>0 else np.nan
            kind='cat' if c in cat_cols else 'num'
            rows.append([fold,c,kind,beta[c],slope])
        LR=2*(ll_d-ll_woe); dfree=k_d-k_woe; lr.append([fold,LR,dfree,stats.chi2.sf(max(LR,0),dfree)])
    sdf=pd.DataFrame(rows,columns=['fold','feature','kind','beta_hat','proj_slope'])
    agg=sdf.groupby(['feature','kind']).agg(beta=('beta_hat','mean'),slope=('proj_slope','mean')).reset_index()
    rr=np.corrcoef(agg['beta'],agg['slope'])[0,1]; gap=(agg['beta']-agg['slope']).abs().mean()
    lrdf=pd.DataFrame(lr,columns=['fold','LR','df','p'])
    print('\n===== %s ====='%name)
    print(agg.round(4).to_string(index=False))
    print('corr(beta,slope)=%.4f | mean|gap|=%.4f | LR median p=%.3g'%(rr,gap,lrdf['p'].median()))
    agg.round(6).to_csv(os.path.join(TABLES,'R1b_%s_slope.csv'%name),index=False)
    return agg, rr, gap

# ===================================================================
# GERMAN
# ===================================================================
cols=['Status_checking','Duration','Credit_history','Purpose','Credit_amount','Savings',
      'Employment_since','Installment_rate','Personal_status_sex','Other_debtors','Residence_since',
      'Property','Age','Other_installment','Housing','Existing_credits','Job','Num_liable',
      'Telephone','Foreign_worker','target']
g=pd.read_csv(os.path.join(DATA,'german.data'),sep=' ',header=None,names=cols)
g['target']=(g['target']==2).astype(int)
gcat=['Status_checking','Credit_history','Purpose','Savings','Employment_since','Personal_status_sex',
      'Other_debtors','Property','Other_installment','Housing','Job','Telephone','Foreign_worker']
gnum=['Duration','Credit_amount','Installment_rate','Residence_since','Age','Existing_credits','Num_liable']
agg_g,rr_g,gap_g=run_dataset('german',g,'target',gcat,gnum,N_SPLIT=5)

# ===================================================================
# HELOC
# ===================================================================
h=pd.read_csv(os.path.join(DATA,'heloc.csv'))
h['RiskPerformance']=(h['RiskPerformance']=='Bad').astype(int)
hfeat=[c for c in h.columns if c!='RiskPerformance']
allmiss=(h[hfeat]==-9).all(axis=1); h=h[~allmiss].reset_index(drop=True)
agg_h,rr_h,gap_h=run_dataset('heloc',h,'RiskPerformance',[],hfeat,N_SPLIT=10)

# ===================================================================
# COMBINED FIGURE: slope vs beta for German + HELOC
# ===================================================================
fig,axes=plt.subplots(1,2,figsize=(11,5.2))
for ax,(agg,rr,nm) in zip(axes,[(agg_g,rr_g,'German'),(agg_h,rr_h,'HELOC')]):
    lo=min(agg['beta'].min(),agg['slope'].min(),0)*1.1; hi=max(agg['beta'].max(),agg['slope'].max())*1.1
    ax.plot([lo,hi],[lo,hi],color='0.0',lw=1,ls='--',label='identity')
    ax.axhline(1.0,color='0.6',lw=1,ls=':',label='naive-Bayes = 1')
    if 'cat' in set(agg['kind']):
        s=agg['kind']=='cat'; ax.scatter(agg[s]['beta'],agg[s]['slope'],s=48,marker='^',c='0.6',edgecolor='0.0',label='categorical')
    s=agg['kind']=='num'; ax.scatter(agg[s]['beta'],agg[s]['slope'],s=48,marker='o',c='0.3',edgecolor='0.0',label='numeric')
    ax.set_title('%s (r=%.3f)'%(nm,rr),fontsize=10)
    ax.set_xlabel(r'logistic-WoE $\hat\beta_j$'); ax.set_ylabel('dummy-logit projection slope')
    ax.legend(frameon=True,facecolor='white',edgecolor='0.5',fontsize=8)
savefig(fig,'R1b_slope_vs_beta_german_heloc')
print('\nSaved figure: R1b_slope_vs_beta_german_heloc')
print('\n================ R1b HEADLINE ================')
print('German: corr=%.3f gap=%.3f | HELOC: corr=%.3f gap=%.3f'%(rr_g,gap_g,rr_h,gap_h))
print('Multivariate slope tracks beta_hat across nominal (German) and informative-missing (HELOC) too;')
print('contrast with naive-Bayes slope = 1. All binning/IV inside folds.')
print('\nR1b complete.')



===== german =====
            feature kind    beta   slope
                Age  num  0.7226  0.7121
      Credit_amount  num  0.9091  1.1397
     Credit_history  cat  0.6620  0.7633
           Duration  num  0.6700  0.7997
   Employment_since  cat  0.6714  0.7080
   Existing_credits  num -0.3903 -0.8575
     Foreign_worker  cat  1.0793  1.1742
            Housing  cat  0.3642  0.1606
   Installment_rate  num  2.1193  2.1215
                Job  cat -0.5644 -0.5543
      Other_debtors  cat  0.9718  1.0648
  Other_installment  cat  0.8552  0.9598
Personal_status_sex  cat  0.9753  0.9796
           Property  cat  0.3219  0.6326
            Purpose  cat  1.1231  1.1956
            Savings  cat  0.7870  0.8624
    Status_checking  cat  0.8328  0.8667
          Telephone  cat  1.2725  1.2685
corr(beta,slope)=0.9731 | mean|gap|=0.1103 | LR median p=0.963



===== heloc =====
                           feature kind    beta   slope
                    AverageMInFile  num  0.5416  0.5607
              ExternalRiskEstimate  num  0.2858  0.2481
              MSinceMostRecentDelq  num  0.4611  0.1059
      MSinceMostRecentInqexcl7days  num  0.6936  0.6770
         MSinceMostRecentTradeOpen  num  0.3308  0.5243
             MSinceOldestTradeOpen  num  0.1568  0.1685
          MaxDelq2PublicRecLast12M  num  0.2262  0.2025
                       MaxDelqEver  num -0.4547 -0.1904
          NetFractionInstallBurden  num  0.0984  0.1307
        NetFractionRevolvingBurden  num  0.3943  0.4167
NumBank2NatlTradesWHighUtilization  num  0.2667  0.2627
                      NumInqLast6M  num  1.3131  1.4508
             NumInqLast6Mexcl7days  num -1.0621 -1.1508
          NumInstallTradesWBalance  num -0.3296 -0.2308
        NumRevolvingTradesWBalance  num  0.3667  0.5025
             NumSatisfactoryTrades  num  0.4895  0.6280
                    NumTotalT


Saved figure: R1b_slope_vs_beta_german_heloc

================ R1b HEADLINE ================
German: corr=0.973 gap=0.110 | HELOC: corr=0.953 gap=0.113
Multivariate slope tracks beta_hat across nominal (German) and informative-missing (HELOC) too;
contrast with naive-Bayes slope = 1. All binning/IV inside folds.

R1b complete.
